<div style="
    background: linear-gradient(90deg, #0f2027 0%, #2c5364 100%);
    padding: 32px 0 32px 0;
    border-radius: 15px;
    text-align: center;
    margin-bottom: 30px;
">
  <h1 style="color: #fff; letter-spacing: 2px; font-size: 2.7rem; margin: 0; font-weight: 800;">
    Module 3: Distributed Training with PyTorch and Ray Train
  </h1>
</div>


This notebook will show you how to distribute your PyTorch training code with Ray Train and Ray Data, getting performance and usability improvements along the way.

In this tutorial, you:
1. Start with a basic single machine PyTorch example.
2. Distribute it to multiple GPUs on multiple machines with [Ray Train](https://docs.ray.io/en/latest/train/train.html) and, if you are using an Anyscale Workspace, inspect results with the Ray Train dashboard.
3. Scale data ingest separately from training with [Ray Data](https://docs.ray.io/en/latest/data/data.html) and, if you are using an Anyscale Workspace, inspect results with the Ray Data dashboard. 

## `Step 1`: Start with a basic single machine PyTorch example

In this step you train a PyTorch VisionTransformer model to recognize objects using the open CIFAR-10 dataset. It's a minimal example that trains on a single machine. Note that the code has multiple functions to highlight the changes needed to run things with Ray.

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/single_gpu_pytorch_v3.png" width="70%" loading="lazy">|
|:--|
|An overview of the single GPU training process. At a high level, here is how training loop in PyTorch looks like. The key stages include loading the dataset; run the training on mini-batches on a single GPU; saving the model checkpoint to the persistent storage.|

In [1]:
import os
from typing import Dict

import torch
from filelock import FileLock
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import Normalize, ToTensor
from torchvision.models import VisionTransformer
from tqdm import tqdm

Next, define a function that returns PyTorch DataLoaders for train and test data. Note how the code also preprocesses the data. 

In [2]:
def get_dataloaders(batch_size):
    # Transform to normalize the input images.
    transform = transforms.Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

    with FileLock(os.path.expanduser("~/data.lock")):
        # Download training data from open datasets.
        training_data = datasets.CIFAR10(
            root="~/data",
            train=True,
            download=True,
            transform=transform,
        )

        # Download test data from open datasets.
        testing_data = datasets.CIFAR10(
            root="~/data",
            train=False,
            download=True,
            transform=transform,
        )

    # Create data loaders.
    train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(testing_data, batch_size=batch_size)

    return train_dataloader, test_dataloader

Now, define a function that runs the core training loop. Note how the code synchronously alternates between training the model for one epoch and then evaluating its performance.

In [3]:
def train_func():
    lr = 1e-3
    epochs = 1
    batch_size = 512

    # Get data loaders inside the worker training function.
    train_dataloader, valid_dataloader = get_dataloaders(batch_size=batch_size)

    model = VisionTransformer(
        image_size=32,   # CIFAR-10 image size is 32x32
        patch_size=4,    # Patch size is 4x4
        num_layers=12,   # Number of transformer layers
        num_heads=8,     # Number of attention heads
        hidden_dim=384,  # Hidden size (can be adjusted)
        mlp_dim=768,     # MLP dimension (can be adjusted)
        num_classes=10   # CIFAR-10 has 10 classes
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    # Model training loop.
    for epoch in range(epochs):
        model.train()
        for X, y in tqdm(train_dataloader, desc=f"Train Epoch {epoch}"):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        valid_loss, num_correct, num_total = 0, 0, 0
        with torch.no_grad():
            for X, y in tqdm(valid_dataloader, desc=f"Valid Epoch {epoch}"):
                X, y = X.to(device), y.to(device)
                pred = model(X)
                loss = loss_fn(pred, y)

                valid_loss += loss.item()
                num_total += y.shape[0]
                num_correct += (pred.argmax(1) == y).sum().item()

        valid_loss /= len(train_dataloader)
        accuracy = num_correct / num_total

        print({"epoch_num": epoch, "loss": valid_loss, "accuracy": accuracy})

Finally, run training.

In [4]:
# train_func()

## `Step 2`: Distribute training to multiple machines with Ray Train



Next, modify this example to run with Ray Train on multiple machines with distributed data parallel (DDP) training. In DDP training, each process trains a copy of the model on a subset of the data and synchronizes gradients across all processes after each backward pass to keep models consistent. Essentially, Ray Train allows you to wrap PyTorch training code in a function and run the function on each worker in your Ray Cluster. With a few modifications, you get the fault tolerance and auto-scaling of a [Ray Cluster](https://docs.ray.io/en/latest/cluster/getting-started.html), as well as the observability and ease-of-use of [Ray Train](https://docs.ray.io/en/latest/train/train.html).

First, set some environment variables and import some modules.

### Architecture overview

Ray Train's architecture is based on the following components:
1. A `Ray Train Controller/Driver` that schedules the training workers, handles errors and manages checkpoints
2. `Ray Train Worker` executing the training code

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/train_architecture.jpg" loading="lazy" width="700">

### Key API concepts

Below are the key API concepts of Ray Train:

1. `train_loop_per_worker`: This is the main function that contains your model training logic.
2. `ScalingConfig`: This is used to specify the number of workers and compute resources (CPUs or GPUs, TPUs).
3. `Trainer`: This is used to manage the training process.
4. `Trainer.fit()`: This is used to start the training process.

<img src="https://docs.ray.io/en/latest/_images/overview.png" width="700" loading="lazy">

Below is the diagram showing how the key components usually work together.

- The Train Controller/Driver is constantly performing health checks on the Train workers.
- The Train workers are running the training loop and at a particular frequency checkpointing the model to a persistent storage.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_train_detailed_architecture.png" width="800" loading="lazy">


Let's consider the case where we have a very large dataset of images that would take a long time to train on a single GPU. We would now like to scale this training job to run on multiple GPUs.

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/multi_gpu_pytorch_v4.png" width="70%" loading="lazy">|
|:--|
|Schematic overview of DistributedDataParallel (DDP) training: (1) the model is replicated from the <code>GPU rank 0</code> to all other workers; (2) each worker receives a shard of the dataset and processes a mini-batch; (3) during the backward pass, gradients are averaged across GPUs; (4) checkpoint and metrics from rank 0 GPU are saved to the persistent storage.|

In [5]:
import ray.train
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

import tempfile
import uuid

Next, modify the training function you wrote earlier. Every difference from the previous script is highlighted and explained with a numbered comment; for example, “[1].”

In [6]:
def train_func_per_worker(config: Dict):
    lr = config["lr"]
    epochs = config["epochs"]
    batch_size = config["batch_size_per_worker"]

    # Step 1: Prepare data loader for distributed training.
    train_dataloader, valid_dataloader = get_dataloaders(batch_size=batch_size)

    train_dataloader = ray.train.torch.prepare_data_loader(train_dataloader)
    valid_dataloader = ray.train.torch.prepare_data_loader(valid_dataloader)

    model = VisionTransformer(
        image_size=32,   # CIFAR-10 image size is 32x32
        patch_size=4,    # Patch size is 4x4
        num_layers=12,   # Number of transformer layers
        num_heads=8,     # Number of attention heads
        hidden_dim=384,  # Hidden size (can be adjusted)
        mlp_dim=768,     # MLP dimension (can be adjusted)
        num_classes=10   # CIFAR-10 has 10 classes
    )

    # Step 2: Prepare and wrap your model with DistributedDataParallel.
    #         The prepare_model method moves the model to the correct GPU/CPU device.
    model = ray.train.torch.prepare_model(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    # Step 3: Model training loop.
    for epoch in range(epochs):
        if ray.train.get_context().get_world_size() > 1:
            # Required for the distributed sampler to shuffle properly across epochs.
            train_dataloader.sampler.set_epoch(epoch)

        model.train()
        for X, y in tqdm(train_dataloader, desc=f"Train Epoch {epoch}"):
            pred = model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        valid_loss, num_correct, num_total = 0, 0, 0
        with torch.no_grad():
            for X, y in tqdm(valid_dataloader, desc=f"Valid Epoch {epoch}"):
                pred = model(X)
                loss = loss_fn(pred, y)

                valid_loss += loss.item()
                num_total += y.shape[0]
                num_correct += (pred.argmax(1) == y).sum().item()

        valid_loss /= len(train_dataloader)
        accuracy = num_correct / num_total

        # Step 4: Report checkpoints and attached metrics to Ray Train.
        # ====================================================================
        metrics = print_metrics_ray_train(loss, epoch)
        save_checkpoint_and_metrics_ray_train(model, metrics)

In [7]:
def print_metrics_ray_train(loss: torch.Tensor, epoch: int) -> None:
    metrics = {"loss": loss.item(), "epoch": epoch}
    world_rank = ray.train.get_context().get_world_rank() # report from all workers
    print(f"{metrics=} {world_rank=}")
    
    return metrics

def save_checkpoint_and_metrics_ray_train(
    model: torch.nn.Module, metrics: dict[str, float]
) -> None:
    with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
        torch.save(
            model.module.state_dict(),  # note the `.module` to unwrap the DistributedDataParallel
            os.path.join(temp_checkpoint_dir, "model.pt"),
        )

        ray.train.report(
            metrics,
            checkpoint=ray.train.Checkpoint.from_directory(temp_checkpoint_dir),
        )

Finally, run the training function on the Ray Cluster with a TorchTrainer using GPU workers.

The `TorchTrainer` takes the following arguments:
* `train_loop_per_worker`: the training function you defined earlier. Each Ray Train worker runs this function. Note that you can call special Ray Train functions from within this function.
* `train_loop_config`: a hyperparameter dictionary. Each Ray Train worker calls its `train_loop_per_worker` function with this dictionary.
* `scaling_config`: a configuration object that specifies the number of workers and compute resources, for example, CPUs or GPUs, that your training run needs.
* `run_config`: a configuration object that specifies various fields used at runtime, including the path to save checkpoints.

`trainer.fit` spawns a controller process to orchestrate the training run and worker processes to actually execute the PyTorch training code.

In [8]:
num_workers=2 
use_gpu=True
global_batch_size = 512
lr = 1e-3
epochs = 2

train_config = {
    "lr": lr,
    "epochs": epochs,
    "batch_size_per_worker": global_batch_size // num_workers,
}

scaling_config = ScalingConfig(num_workers=num_workers, 
                               use_gpu=use_gpu)

run_config = RunConfig(storage_path="/mnt/cluster_storage", 
                       name=f"train_run-{uuid.uuid4().hex}")

trainer = TorchTrainer(
    train_loop_per_worker=train_func_per_worker,
    train_loop_config=train_config,
    scaling_config=scaling_config,
    run_config=run_config,
)

result = trainer.fit()
print(f"Training result: {result}")

2026-03-10 04:18:00,082	INFO worker.py:1810 -- Connecting to existing Ray cluster at address: 10.0.152.157:6379...
2026-03-10 04:18:00,109	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at https://session-w9hutqjs6uvhf2vxxemg8rde8q.i.anyscaleuserdata.com 
2026-03-10 04:18:00,114	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_3a7b63e92805e675ce40c0b2acc308ee90e989dd.zip' (1.44MiB) to Ray cluster...
2026-03-10 04:18:00,121	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_3a7b63e92805e675ce40c0b2acc308ee90e989dd.zip'.
/home/ray/anaconda3/lib/python3.13/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(TrainController pid=257713) [State Transition] INITIALIZING -> SCHEDULI

Training result: Result(metrics={'loss': 1.5540642738342285, 'epoch': 1}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/train_run-fb764d64f50843358192dc5e88ec624c/checkpoint_2026-03-10_04-19-27.562301), error=None, path='/mnt/cluster_storage/train_run-fb764d64f50843358192dc5e88ec624c', metrics_dataframe=       loss  epoch
0  1.790780      0
1  1.554064      1, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/train_run-fb764d64f50843358192dc5e88ec624c/checkpoint_2026-03-10_04-18-49.790122), {'loss': 1.7907798290252686, 'epoch': 0}), (Checkpoint(filesystem=local, path=/mnt/cluster_storage/train_run-fb764d64f50843358192dc5e88ec624c/checkpoint_2026-03-10_04-19-27.562301), {'loss': 1.5540642738342285, 'epoch': 1})], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x701204d4d670>)


Because you're running training in a data parallel fashion this time, it should take under 1 minute while maintaining similar accuracy.

Distributed data-parallel training, but now using Ray Train.

|<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/multi_gpu_pytorch_annotated_v5.png" width="70%" loading="lazy">|
|:--|
||

## `Step 3`: Scale data ingest separately from training with Ray Data




Modify this example to load data with Ray Data instead of the native PyTorch DataLoader. With a few modifications, you can scale data preprocessing and training separately. For example, you can do the former with a pool of CPU workers and the latter with a pool of GPU workers. See [How does Ray Data compare to other solutions for offline inference?](https://docs.ray.io/en/latest/data/comparisons.html#how-does-ray-data-compare-to-other-solutions-for-ml-training-ingest) for a comparison between Ray Data and PyTorch data loading.

First, create [Ray Data Datasets](https://docs.ray.io/en/latest/data/key-concepts.html#datasets-and-blocks) from S3 data and inspect their schemas.

### Compare differences on a distributed setup

Let's now look at how PyTorch DataLoader vs Ray Data work in a distributed setup.


Below is a diagram showing how Pytorch DataLoader works in a distributed setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/pytorch_dataloader_multi_worker.png" width="800" loading="lazy">

Below is a diagram showing how Ray Data works in a distributed setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_data_ray_train_multi_worker.png" width="1000" loading="lazy">


### When to Integrate Ray Train with Ray Data
Use both Ray Train and Ray Data when you face one of the following challenges:
| Challenge | Detail | Solution |
| --- | --- | --- |
| Need a **consistent interface for loading and sharding** data | The training process may need to load data from various sources, such as Parquet, CSV, or lakehouses. | Ray Data provides a standardize approach for loading, shuffling, sharding, and streaming batch data to your distributed training run. |
| Need to perform online or **just-in-time data processing** on **CPU and GPU nodes** | You are in a scenario where it makes more sense to perform data preprocessing on the fly instead of preprocess your data in batch in a separate job. This usually is the case if preprocessing your data takes an extremely long time (on the order of weeks) and you don't want this to block your training run. | Ray Train's integration with Ray Data makes it easy to implement just-in-time data processing. |

### Integration Architecture

Here is a diagram showing the a sample Ray Data and Ray Train integration for a simple homogenous cluster setup

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/homogeneous_training_ray_train_ray_data.png" width="900" loading="lazy">

## Ray Train in Production with Ray Data

Here is how our training loop will look like using **Ray Data** instead of the **PyTorch DataLoader**:

In [9]:
import ray.data

import numpy as np

STORAGE_PATH = "s3://ray-example-data/cifar10-parquet"

train_dataset = ray.data.read_parquet(f'{STORAGE_PATH}/train')
test_dataset = ray.data.read_parquet(f'{STORAGE_PATH}/test')

train_dataset.schema()
test_dataset.schema()

(TrainController pid=257713) [State Transition] SHUTTING_DOWN -> FINISHED.


Column  Type
------  ----
image   ArrowTensorType(shape=(32, 32, 3), dtype=uint8)
label   int64

Next, use Ray Data to transform the data. Note that both loading and transformation happen lazily, which means that only the training workers materialize the data.

In [10]:
def transform_cifar(row: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    # Define the torchvision transform.
    transform = transforms.Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    row["image"] = transform(row["image"])
    return row

train_dataset = train_dataset.map(transform_cifar)
test_dataset = test_dataset.map(transform_cifar)

Next, modify the training function you wrote earlier. Every difference from the previous script is highlighted and explained with a numbered comment; for example, “[1].”

In [11]:
def train_func_per_worker(config: Dict):
    lr = config["lr"]
    epochs = config["epochs"]
    batch_size = config["batch_size_per_worker"]


    # [1] Prepare `Dataloader` for distributed training.
    # The get_dataset_shard method gets the associated dataset shard to pass to the 
    # TorchTrainer constructor in the next code block.
    # The iter_torch_batches method lazily shards the dataset among workers.
    # =============================================================================
    train_data_shard = ray.train.get_dataset_shard("train")
    valid_data_shard = ray.train.get_dataset_shard("valid")
    train_dataloader = train_data_shard.iter_torch_batches(batch_size=batch_size)
    valid_dataloader = valid_data_shard.iter_torch_batches(batch_size=batch_size)

    model = VisionTransformer(
        image_size=32,   # CIFAR-10 image size is 32x32
        patch_size=4,    # Patch size is 4x4
        num_layers=12,   # Number of transformer layers
        num_heads=8,     # Number of attention heads
        hidden_dim=384,  # Hidden size (can be adjusted)
        mlp_dim=768,     # MLP dimension (can be adjusted)
        num_classes=10   # CIFAR-10 has 10 classes
    )

    model = ray.train.torch.prepare_model(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    # Model training loop.
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_dataloader, desc=f"Train Epoch {epoch}"):
            X, y = batch['image'], batch['label']
            pred = model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        valid_loss, num_correct, num_total, num_batches = 0, 0, 0, 0
        with torch.no_grad():
            for batch in tqdm(valid_dataloader, desc=f"Valid Epoch {epoch}"):
                # [2] Each Ray Data batch is a dict so you must access the
                # underlying data using the appropriate keys.
                # =======================================================
                X, y = batch['image'], batch['label']
                pred = model(X)
                loss = loss_fn(pred, y)

                valid_loss += loss.item()
                num_total += y.shape[0]
                num_batches += 1
                num_correct += (pred.argmax(1) == y).sum().item()

        valid_loss /= num_batches
        accuracy = num_correct / num_total

        metrics = print_metrics_ray_train(loss,  epoch)
        save_checkpoint_and_metrics_ray_train(model, metrics)


Finally, run the training function with the Ray Data Dataset on the Ray Cluster with 8 GPU workers.

In [12]:
def train_cifar_10(num_workers, use_gpu):
    global_batch_size = 512

    train_config = {
        "lr": 1e-3,
        "epochs": 1,
        "batch_size_per_worker": global_batch_size // num_workers,
    }

    # Configure computation resources.
    scaling_config = ScalingConfig(num_workers=num_workers, use_gpu=use_gpu)
    run_config = RunConfig(
        storage_path="/mnt/cluster_storage", 
        name=f"train_data_run-{uuid.uuid4().hex}",
    )

    # Initialize a Ray TorchTrainer.
    trainer = TorchTrainer(
        train_loop_per_worker=train_func_per_worker,
        # [1] With Ray Data you pass the Dataset directly to the Trainer.
        # ==============================================================
        datasets={"train": train_dataset, "valid": test_dataset},
        train_loop_config=train_config,
        scaling_config=scaling_config,
        run_config=run_config,
    )

    result = trainer.fit()
    print(f"Training result: {result}")

if __name__ == "__main__":
    train_cifar_10(num_workers=2, use_gpu=True)

(TrainController pid=258579) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=258579) Attempting to start training worker group of size 2 with the following resources: [{'GPU': 1}] * 2
(RayTrainWorker pid=76801, ip=10.0.170.113) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=258579) Started training worker group of size 2: 
(TrainController pid=258579) - (ip=10.0.170.113, pid=76801) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=258579) - (ip=10.0.184.0, pid=75523) world_rank=1, local_rank=0, node_rank=1
(TrainController pid=258579) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=75523, ip=10.0.184.0) Moving model to device: cuda:0
(RayTrainWorker pid=75523, ip=10.0.184.0) Wrapping provided model in DistributedDataParallel.
Train Epoch 0: 0it [00:00, ?it/s]p=10.0.184.0) 
(SplitCoordinator pid=258878) Registered dataset logger for dataset train_37_0
(SplitCoordinator pid=258878) Starting execution of Datase

(pid=258878) Running Dataset train_37_0.: 0.00 row [00:00, ? row/s]

(pid=258878) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=258878) - ReadFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=258878) - Map(transform_cifar):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=258926) - Map(transform_cifar):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=258926) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=258926) Registered dataset logger for dataset valid_39_0
(SplitCoordinator pid=258926) Starting execution of Dataset valid_39_0. Full logs are in /tmp/ray/session_2026-03-09_18-40-11_574972_2359/logs/ray-data
(SplitCoordinator pid=258926) Execution plan of Dataset valid_39_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[Map(transform_cifar)] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=258926) ⚠️  Ray's object store is configured to use only 28.0% of available memory (26.8GiB out of 96.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=258926) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext

Training result: Result(metrics={'loss': 1.6885604858398438, 'epoch': 0}, checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/train_data_run-8d07ece440e246f197c4cb9cfe251e09/checkpoint_2026-03-10_04-20-55.616724), error=None, path='/mnt/cluster_storage/train_data_run-8d07ece440e246f197c4cb9cfe251e09', metrics_dataframe=      loss  epoch
0  1.68856      0, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/cluster_storage/train_data_run-8d07ece440e246f197c4cb9cfe251e09/checkpoint_2026-03-10_04-20-55.616724), {'loss': 1.6885604858398438, 'epoch': 0})], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x701204c68830>)


(TrainController pid=258579) [State Transition] SHUTTING_DOWN -> FINISHED.


#### Note on the checkpoint lifecycle

Here is the lifecycle of a checkpoint from being created using a local path to being uploaded to persistent storage.

<img src="https://docs.ray.io/en/latest/_images/checkpoint_lifecycle.png" width=800>


<div class="alert alert-block alert-info">
  <p><strong>Notes:</strong></p>
  <ul>
    <li>
      Given it is the same model across all workers, we can instead only build the checkpoint on the worker of rank 0.
      Note that we will still need to call 
      <a href="https://docs.ray.io/en/latest/train/api/doc/ray.train.report.html#ray.train.report" target="_blank">
        ray.train.report
      </a> on all workers to ensure that the training loop is synchronized.
    </li>
    <li>Ray Train expects all workers to be able to write files to the same persistent storage location.</li>
    <li>Cloud storage is the recommended persistent storage location.</li>
  </ul>
</div>